In [1]:
import os
from pathlib import Path

for root in sorted(Path("/kaggle/input").glob("*")):
    print("===", root.name)
    count = 0
    with os.scandir(root) as entries:
        for entry in sorted(entries, key=lambda e: e.name):
            kind = "dir " if entry.is_dir() else "file"
            print(f"    {kind} {entry.name}")
            count += 1
            if count >= 15:
                print("    ... more not shown")
                break

=== competitions
    dir  rsna-knee-abnormality-detection
=== datasets
    dir  mohammedtalafha


In [2]:
import sys
from pathlib import Path

BASE = Path("/kaggle/input")
MINE = BASE / "datasets" if (BASE / "datasets").is_dir() else BASE
COMP = BASE / "competitions" / "rsna-knee-abnormality-detection"

# your code
CODE_ROOT = None
for marker in MINE.rglob("architecture.py"):
    if marker.parent.name == "model":
        CODE_ROOT = marker.parent.parent
        break
if CODE_ROOT is None:
    raise FileNotFoundError("could not find model/architecture.py in your datasets")

# your model
MODEL_PATHS = sorted(MINE.rglob("*.pt"))
if not MODEL_PATHS:
    raise FileNotFoundError("no .pt file found in your datasets")
MODEL_PATH = MODEL_PATHS[0]

# the competition data (top level only -- fast)
DATA_ROOT = COMP if (COMP / "test.csv").is_file() else None
if DATA_ROOT is None:
    for child in sorted(COMP.glob("*")):
        if child.is_dir() and (child / "test.csv").is_file():
            DATA_ROOT = child
            break
if DATA_ROOT is None:
    print("test.csv not found. Competition folder holds:")
    for item in sorted(COMP.glob("*"))[:20]:
        print("   ", "dir " if item.is_dir() else "file", item.name)
    raise FileNotFoundError("could not find test.csv")

sys.path.insert(0, str(CODE_ROOT))
sys.path.insert(0, str(CODE_ROOT / "developments" / "src"))
print("code :", CODE_ROOT)
for path in MODEL_PATHS:
    print("model:", path)
print("data :", DATA_ROOT)

code : /kaggle/input/datasets/mohammedtalafha/cnn-cpc-code
model: /kaggle/input/datasets/mohammedtalafha/frozen-all-lang/model_finetuned.pt
model: /kaggle/input/datasets/mohammedtalafha/frozen-all-lang/model_frozen.pt
data : /kaggle/input/competitions/rsna-knee-abnormality-detection


In [3]:
from model._implementation import read_config
from model.architecture import load

config = read_config(str(CODE_ROOT / "config" / "current_model.yaml"))
config["data_root"] = str(DATA_ROOT)

model, payload = load(str(MODEL_PATH), device="cpu")
print("encoder     :", payload.get("encoder_source", "report-aligned"))
print("epochs done :", payload.get("completed_epochs"))

if str(payload.get("encoder_source", "report-aligned")) == "dinov3":
    raise RuntimeError("this is the DINOv3 model -- it needs internet. Use the other one.")

del model

encoder     : report-aligned
epochs done : 2


In [4]:
from testing.test import predict_test_set

out = predict_test_set(
    config,
    checkpoint=str(MODEL_PATH),
    out_path="/kaggle/working/submission.csv",
)
print("wrote", out)

device=Tesla T4 | single-gpu | precision=fp16 | workers=2 | visible_gpus=2
[ensemble] loaded model_finetuned.pt from frozen-all-lang
{
  "checkpoints": [
    "/kaggle/input/datasets/mohammedtalafha/frozen-all-lang/model_finetuned.pt"
  ],
  "ensemble_size": 1,
  "checkpoint": "/kaggle/input/datasets/mohammedtalafha/frozen-all-lang/model_finetuned.pt",
  "checkpoint_sha256": "43bb078790c0b8c560bd60c6e07f5294b35231b420be609625c34fe89225c6aa",
  "checkpoint_sha256_all": [
    "43bb078790c0b8c560bd60c6e07f5294b35231b420be609625c34fe89225c6aa"
  ],
  "submission_sha256": "3b7880d6372161b0790cdd8872e72218870f851d1a606e99f299c93578e16879",
  "completed_epochs": 2,
  "fixed_endpoint": true,
  "encoder_frozen": true,
  "encoder_sha256": "7c63f445157e44a6f5ca7785f51671b2af92d5b77edae65427935d0993abd058",
  "expert_labels_in_gradients": 0,
  "crop_policy": {
    "version": "joint_focus_center_crop_only_v1",
    "crop_fraction": 0.9
  },
  "slice_offsets": [
    -1,
    0,
    1
  ],
  "test_studi

In [5]:
import pandas as pd

frame = pd.read_csv("/kaggle/working/submission.csv")
print("rows   :", len(frame))
print("columns:", len(frame.columns))
print(frame.head())

scores = frame.drop(columns=["StudyInstanceUID"])
spread = (scores.max() - scores.min()).sort_values()
print("\nsmallest spreads:")
print(spread.head(3))
if spread.max() < 0.01:
    print("\nWARNING: every column is nearly the same -- do not submit this")

rows   : 3
columns: 13
                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.224088  0.177253   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.406506  0.294254   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.177380  0.150449   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.617994          0.103650   0.556536    0.432930  0.439064   
1         0.777126          0.150905   0.780988    0.720374  0.659682   
2         0.548731          0.090186   0.475295    0.336340  0.372184   

   Effusion  Synovitis   Baker's  Contusion  Fracture  
0  0.351418   0.894600  0.324335   0.333040  0.257433  
1  0.617207   0.910053  0.643525   0.627568  0.609219  
2  0.269854   0.887531  0.248601   0.244678  0.182903  

smallest spreads:
Synovitis           0.022522
Lateral Meniscus    0.060719
MCL                 0.143805
dtype: float64


In [6]:
import torch

p = torch.load(str(MODEL_PATH), map_location="cpu", weights_only=False)
print("file       :", MODEL_PATH.name)
print("before     :", p.get("encoder_sha256_initial"))
print("after      :", p.get("encoder_sha256_final"))
print("moved      :", p.get("encoder_sha256_initial") != p.get("encoder_sha256_final"))
print("stages     :", p.get("encoder_trainable_stages"))
print("all models :", [q.name for q in MODEL_PATHS])

file       : model_finetuned.pt
before     : b328667cf9dfa9b909ef181c1bcc8975ec42bcd8b9eddad08f908875b73fae96
after      : 7c63f445157e44a6f5ca7785f51671b2af92d5b77edae65427935d0993abd058
moved      : True
stages     : 1
all models : ['model_finetuned.pt', 'model_frozen.pt']
